# Librerias

In [ ]:
import pandas as pd
import numpy as np
from kmodes.kprototypes import KPrototypes
import matplotlib.pyplot as plt

# Carga de la bbdd y reducción

## 1. Cargar csv

In [4]:
#cargar csv
df = pd.read_csv("../data/df_original_260420.csv")

# Nos quedamos con las variables que usaremos para segmentar
cols = ["age", "job", "marital", "education", "balance", "deposit", "housing", "loan"]

df_perfil = df[cols].copy()
df_perfil

,age,job,marital,education,balance,deposit,housing,loan
0,59.0,admin.,married,secondary,2343,yes,yes,no
1,56.0,admin.,married,secondary,45,yes,no,no
2,41.0,technician,married,secondary,1270,yes,yes,no
3,55.0,services,married,secondary,2476,yes,yes,no
4,54.0,admin.,married,tertiary,184,yes,no,no
...,...,...,...,...,...,...,...,...
10637,33.0,technician,married,secondary,218,no,yes,yes
10638,42.0,management,single,tertiary,1146,no,yes,no
10639,31.0,unemployed,single,unknown,167,no,no,no
10640,30.0,blue-collar,single,secondary,447,no,no,no


## 2. Categorizar edad (tal vez ya está)

In [ ]:
df_perfil["age_group"] = pd.cut(df_perfil["age"], bins=[0, 25, 35, 45, 55, 65, 120], labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"])

#Eliminamos la columna numérica original
df_perfil = df_perfil.drop(columns=["age"])

## 3. Convertir variables a 0 y 1 (tal vez ya está)

In [6]:
for col in ["deposit", "housing", "loan"]:
    df_perfil[col] = df_perfil[col].map({"yes": 1, "no": 0})

In [7]:
df_perfil

,job,marital,education,balance,deposit,housing,loan,age_group
0,admin.,married,secondary,2343,1,1,0,56-65
1,admin.,married,secondary,45,1,0,0,56-65
2,technician,married,secondary,1270,1,1,0,36-45
3,services,married,secondary,2476,1,1,0,46-55
4,admin.,married,tertiary,184,1,0,0,46-55
...,...,...,...,...,...,...,...,...
10637,technician,married,secondary,218,0,1,1,26-35
10638,management,single,tertiary,1146,0,1,0,36-45
10639,unemployed,single,unknown,167,0,0,0,26-35
10640,blue-collar,single,secondary,447,0,0,0,26-35


# Perfil del cliente

## 1. Preparar datos para K‑Prototipos

In [8]:
#Dividir categóricas y numéricas
cat_cols = ["age_group", "job", "marital", "education"]
num_cols = ["deposit", "housing", "loan"]

#Matriz que usará K‑Prototipos, reordenado por tipo de variable
data_model = df_perfil[num_cols + cat_cols].copy()

#Guardamos los índices de las columnas categóricas, para saber que se trata como categórica
categorical_indices = [data_model.columns.get_loc(col) for col in cat_cols]

#Convertimos a numpy
X_proto = data_model.to_numpy()

## 2. Método del codo para elegir k

In [9]:
costs = []
ks = list(range(2, 10))

for k in ks:
    kproto = KPrototypes(n_clusters=k, init='Cao', verbose=0, random_state=42)
    clusters = kproto.fit_predict(X_proto, categorical=categorical_indices)
    costs.append(kproto.cost_)

# Graficar la curva del codo, el “codo” es donde la curva deja de bajar bruscamente, ese será k óptimo
plt.figure(figsize=(8, 5))
plt.plot(ks, costs, marker='o')
plt.title("Método del Codo para K-Prototypes")
plt.xlabel("Número de Clusters (k)")
plt.ylabel("Costo")
plt.grid(True)
plt.show()

ValueError: Input contains NaN

## 3. Entrenar K‑Prototipos

In [ ]:
#Entrenar K‑Prototipos
k_opt = 4

kproto = KPrototypes(n_clusters=k_opt, init='Cao', verbose=0, random_state=42)
clusters = kproto.fit_predict(X_proto, categorical=categorical_indices)

#Guardamos el cluster en el dataframe original
df_perfil["cluster"] = clusters

## 3. Ver los prototipos de cada cluster

In [ ]:
#Prototipos (centros) de cada cluster
prototypes = kproto.cluster_centroids_

num_prototypes = prototypes[0]  # numéricas
cat_prototypes = prototypes[1]  # categóricas

print("Centros numéricos:\n", num_prototypes)
print("Centros categóricos:\n", cat_prototypes)

## 4. Perfil de productos por cluster

In [ ]:
#Para identificar que cluster destaca por producto:
cluster_products = df_perfil.groupby("cluster")[["deposit","housing","loan"]].mean()

print(cluster_products)

## 5. Perfil demográfico por cluster

In [ ]:
#Distribución de age_group:
cluster_age = pd.crosstab(df_perfil["cluster"], df_perfil["age_group"], normalize="index")

print(cluster_age)

In [ ]:
#Distribución de job:
cluster_job = pd.crosstab(df_perfil["cluster"], df_perfil["job"], normalize="index")

print(cluster_job)

In [ ]:
#Distribución de marital:
cluster_marital = pd.crosstab(df_perfil["cluster"], df_perfil["marital"], normalize="index")

print(cluster_marital)

In [ ]:
#Distribución de education:
cluster_education = pd.crosstab(df_perfil["cluster"], df_perfil["education"], normalize="index")

print(cluster_education)

## 6. Conclusiones